In [1]:
import duckdb
con=duckdb.connect()

In [2]:
cols = ["CMPLID","ODINO","MFR_NAME","MAKETXT","MODELTXT","YEARTXT","CRASH","FAILDATE",
        "FIRE","INJURED","DEATHS","COMPDESC","CITY","STATE","VIN","DATEA","LDATE","MILES",
        "OCCURENCES","CDESCR","CMPL_TYPE","POLICE_RPT_YN","PURCH_DT","ORIG_OWNER_YN",
        "ANTI_BRAKES_YN","CRUISE_CONT_YN","NUM_CYLS","DRIVE_TRAIN","FUEL_SYS","FUEL_TYPE",
        "TRANS_TYPE","VEH_SPEED","DOT","TIRE_SIZE","LOC_OF_TIRE","TIRE_FAIL_TYPE",
        "ORIG_EQUIP_YN","MANUF_DT","SEAT_TYPE","RESTRAINT_TYPE","DEALER_NAME","DEALER_TEL",
        "DEALER_CITY","DEALER_STATE","DEALER_ZIP","PROD_TYPE","REPAIRED_YN","MEDICAL_ATTN",
        "VEHICLES_TOWED_YN","STATE_OF_INCIDENT","VEHICLE_OPERATOR"]

len(cols)

51

In [3]:
path = "../data/bulk/COMPLAINTS_RECEIVED_2025-2026.txt"

con.execute(f"""
    CREATE OR REPLACE VIEW complaints_2526 AS
    SELECT * FROM read_csv(
        '{path}',
        delim = '\t',
        header = false,
        names = {cols},
        all_varchar = true
    )
""")

Reconciliation check: row count matches the awk count (207,099), so no rows were dropped on load.

In [4]:
con.sql("SELECT COUNT(*) FROM complaints_2526")

count_star()BIGINT
207099


Finding: ODINO repeats across components (see 11633475). Count distinct ODINO, not rows.

In [5]:
con.sql("SELECT ODINO, MAKETXT, MODELTXT, YEARTXT, COMPDESC, LDATE FROM complaints_2526 LIMIT 5")

ODINOVARCHAR,MAKETXTVARCHAR,MODELTXTVARCHAR,YEARTXTVARCHAR,COMPDESCVARCHAR,LDATEVARCHAR
11633472,FORD,EXPLORER,2016,STRUCTURE:BODY,20250101
11633473,JEEP,GLADIATOR,2022,STEERING,20250101
11633474,TOYOTA,TUNDRA,2024,SERVICE BRAKES,20250101
11633475,BUICK,ENVISTA,2024,ELECTRICAL SYSTEM,20250101
11633475,BUICK,ENVISTA,2024,VEHICLE SPEED CONTROL,20250101


In [6]:
paths = [
    "../data/bulk/COMPLAINTS_RECEIVED_2020-2024.txt",
    "../data/bulk/COMPLAINTS_RECEIVED_2025-2026.txt",
]

con.execute(f"""
    CREATE OR REPLACE VIEW complaints AS
    SELECT * FROM read_csv(
        {paths},
        delim = '\t',
        header = false,
        names = {cols},
        all_varchar = true,
        filename = true
    )
""")

In [7]:
con.sql("SELECT filename, COUNT(*) FROM complaints GROUP BY filename")

filenameVARCHAR,count_star()BIGINT
../data/bulk/COMPLAINTS_RECEIVED_2020-2024.txt,418884
../data/bulk/COMPLAINTS_RECEIVED_2025-2026.txt,207099


## 1. Quick profile
**Why:** before any analysis, check the dataset's size, what each column contains, and how much is missing. This decides which columns are worth keeping.

In [ ]:
# shape: rows x columns
n_rows = con.sql("SELECT COUNT(*) FROM complaints").fetchall()[0][0]
n_cols = len(con.sql("SELECT * FROM complaints LIMIT 0").columns)
print(f"shape: {n_rows:,} rows x {n_cols} columns")

# profile every column
profile = con.sql("SUMMARIZE complaints").df()
profile[["column_name", "approx_unique", "null_percentage", "min", "max"]]

AttributeError: Relations no longer have fetchone(): a relation keeps no result between calls, so every call runs it again. Use rel.limit(1).fetchall() instead.